# Data Exploration - OpenSanctions Default Dataset

**Objective**: Understand the structure, content, and characteristics of the OpenSanctions dataset.

**Dataset**: targets.nested.json (3.68 GB) and targets.simple.csv (431.88 MB)

**Key Questions**:
1. What entity types exist? (People, Companies, Vessels, etc.)
2. What fields are available? (name, alias, previousName, description, etc.)
3. What is the field coverage? (missing values, data quality)
4. What languages and transliterations are present?
5. How complex is the nested structure?

## Setup

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Explore Simple CSV Format First

Start with the simpler CSV format to get a quick overview.

In [ ]:
# Load simple CSV (smaller, easier to explore)
csv_path = Path('../data/raw/targets.simple.csv')

if csv_path.exists():
    print(f"Loading {csv_path}...")
    df_simple = pd.read_csv(csv_path, low_memory=False)
    print(f"✓ Loaded {len(df_simple):,} rows")
else:
    print(f"❌ File not found: {csv_path}")
    print("Please ensure data is downloaded to data/raw/")

In [ ]:
# Display first few rows
df_simple.head()

In [ ]:
# Dataset shape
print(f"Shape: {df_simple.shape}")
print(f"Rows: {df_simple.shape[0]:,}")
print(f"Columns: {df_simple.shape[1]}")

In [ ]:
# Column names and types
print("\nColumn Information:")
df_simple.info()

## 2. Entity Type Distribution

In [ ]:
# Count entity types
if 'schema' in df_simple.columns:
    entity_types = df_simple['schema'].value_counts()
    print("Entity Type Distribution:")
    print(entity_types)
    
    # Visualize
    plt.figure(figsize=(12, 6))
    entity_types.plot(kind='bar', color='steelblue')
    plt.title('Entity Type Distribution', fontsize=16)
    plt.xlabel('Entity Type', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 3. Field Coverage Analysis

In [ ]:
# Missing values analysis
missing_stats = pd.DataFrame({
    'Missing Count': df_simple.isnull().sum(),
    'Missing %': (df_simple.isnull().sum() / len(df_simple) * 100).round(2),
    'Present %': ((1 - df_simple.isnull().sum() / len(df_simple)) * 100).round(2)
})

missing_stats = missing_stats.sort_values('Present %', ascending=False)
print("\nField Coverage (sorted by presence):")
print(missing_stats)

In [ ]:
# Visualize field coverage
plt.figure(figsize=(14, 8))
present_pct = missing_stats['Present %'].sort_values(ascending=True)
present_pct.plot(kind='barh', color='seagreen')
plt.title('Field Coverage (% Present)', fontsize=16)
plt.xlabel('Percentage Present', fontsize=12)
plt.ylabel('Field', fontsize=12)
plt.xlim(0, 100)
plt.axvline(50, color='red', linestyle='--', alpha=0.3, label='50% threshold')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Key Fields for IR System

In [ ]:
# Identify key text fields for retrieval
text_fields = ['name', 'alias', 'previousName', 'description', 'summary', 'topics']
identifier_fields = ['imoNumber', 'mmsi', 'callSign', 'registrationNumber']
metadata_fields = ['schema', 'country', 'programId', 'flag']

print("Key Field Analysis:\n")

# Text fields
print("TEXT FIELDS (for retrieval):")
for field in text_fields:
    if field in df_simple.columns:
        non_null = df_simple[field].notna().sum()
        pct = (non_null / len(df_simple) * 100)
        print(f"  {field:20s}: {non_null:>8,} ({pct:>5.1f}%)")
    else:
        print(f"  {field:20s}: NOT FOUND")

print("\nIDENTIFIER FIELDS (for exact match):")
for field in identifier_fields:
    if field in df_simple.columns:
        non_null = df_simple[field].notna().sum()
        pct = (non_null / len(df_simple) * 100)
        print(f"  {field:20s}: {non_null:>8,} ({pct:>5.1f}%)")
    else:
        print(f"  {field:20s}: NOT FOUND")

print("\nMETADATA FIELDS (for filtering):")
for field in metadata_fields:
    if field in df_simple.columns:
        non_null = df_simple[field].notna().sum()
        pct = (non_null / len(df_simple) * 100)
        print(f"  {field:20s}: {non_null:>8,} ({pct:>5.1f}%)")
    else:
        print(f"  {field:20s}: NOT FOUND")

## 5. Sample Records by Entity Type

In [ ]:
# Show examples of different entity types
if 'schema' in df_simple.columns:
    for entity_type in df_simple['schema'].value_counts().head(5).index:
        print(f"\n{'='*80}")
        print(f"SAMPLE: {entity_type}")
        print('='*80)
        sample = df_simple[df_simple['schema'] == entity_type].head(1)
        
        # Display non-null fields
        for col in sample.columns:
            val = sample[col].iloc[0]
            if pd.notna(val):
                print(f"{col:20s}: {val}")

## 6. Explore Nested JSON Structure

Now let's look at the nested JSON format to understand relationships and complex structures.

In [ ]:
# Load first 100 records from nested JSON for exploration
json_path = Path('../data/raw/targets.nested.json')

if json_path.exists():
    print(f"Loading sample from {json_path}...")
    
    sample_records = []
    with open(json_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 100:  # Load only 100 for quick exploration
                break
            try:
                record = json.loads(line)
                sample_records.append(record)
            except json.JSONDecodeError:
                print(f"Error parsing line {i}")
    
    print(f"✓ Loaded {len(sample_records)} sample records")
else:
    print(f"❌ File not found: {json_path}")

In [ ]:
# Examine structure of first record
if sample_records:
    print("Structure of first record:")
    print(json.dumps(sample_records[0], indent=2)[:2000])  # First 2000 chars

In [ ]:
# Analyze nested structure
def analyze_structure(record, prefix=''):
    """Recursively analyze nested structure"""
    fields = {}
    for key, value in record.items():
        field_name = f"{prefix}.{key}" if prefix else key
        
        if isinstance(value, dict):
            fields.update(analyze_structure(value, field_name))
        elif isinstance(value, list):
            fields[field_name] = f"list[{len(value)}]"
            if value and isinstance(value[0], dict):
                fields.update(analyze_structure(value[0], f"{field_name}[0]"))
        else:
            fields[field_name] = type(value).__name__
    
    return fields

if sample_records:
    # Analyze first record
    structure = analyze_structure(sample_records[0])
    print("\nField Structure (nested):")
    for field, dtype in sorted(structure.items()):
        print(f"  {field:50s}: {dtype}")

## 7. Language and Transliteration Analysis

In [ ]:
# Analyze name variations and aliases
if 'name' in df_simple.columns and 'alias' in df_simple.columns:
    print("Name Variation Analysis:\n")
    
    # Count entities with aliases
    has_alias = df_simple['alias'].notna().sum()
    print(f"Entities with aliases: {has_alias:,} ({has_alias/len(df_simple)*100:.1f}%)")
    
    # Sample entities with multiple aliases (if stored as delimited)
    alias_sample = df_simple[df_simple['alias'].notna()].head(10)
    print("\nSample entities with aliases:")
    for idx, row in alias_sample.iterrows():
        print(f"  Name: {row['name']}")
        print(f"  Alias: {row['alias']}")
        print()

## 8. Jurisdiction and Source Analysis

In [ ]:
# Analyze programId/source distribution
if 'programId' in df_simple.columns:
    program_dist = df_simple['programId'].value_counts().head(20)
    print("Top 20 Sanction Programs/Sources:")
    print(program_dist)
    
    # Visualize
    plt.figure(figsize=(12, 8))
    program_dist.plot(kind='barh', color='coral')
    plt.title('Top 20 Sanction Programs/Sources', fontsize=16)
    plt.xlabel('Count', fontsize=12)
    plt.ylabel('Program ID', fontsize=12)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 9. Key Findings Summary

Document key observations for the implementation phases.

### Findings:

**Data Structure:**
- [To be filled after running notebook]

**Field Coverage:**
- [Key fields and their coverage]

**Data Quality Issues:**
- [Missing values, inconsistencies]

**Complexity Assessment:**
- [Nested structures, relationships]

**Language Diversity:**
- [Transliteration challenges]

**Implications for IR System:**
- [What this means for indexing and retrieval]

## Next Steps

1. Build streaming JSON parser (Phase 2, Step 4)
2. Implement text preprocessing pipeline (Phase 2, Step 5)
3. Create document flattening logic (Phase 2, Step 6)
4. Test on 100K subset (Phase 2, Step 7)